#Ranking Analysis
#####Purpose:
- To rank items (e.g., products, customers) based on performance or other metrics.
- To identify top performers or laggards.
	
#####SQL Functions Used:
- Window Ranking Functions: RANK(), DENSE_RANK(), ROW_NUMBER(), TOP
- Clauses: GROUP BY, ORDER BY
##### Rank [Dimension] By Aggregated [Measure]
- Rank Countries By Total Sales
- Top 5 Products By Quantity
- Bottom 3 Customers By Total Orders

####Which 5 products Generating the Highest Revenue?
- Simple Ranking

In [0]:
query = """
SELECT
    p.product_name,
    SUM(f.sales_amount) AS total_renenue
FROM gold.fact_sales AS f
LEFT JOIN gold.dim_products AS p
    ON p.product_key = f.product_key
GROUP BY p.product_name
ORDER BY total_renenue DESC
LIMIT 5
"""
df = spark.sql(query)

df.display()

product_name,total_renenue
Mountain-200 Black- 46,1373454
Mountain-200 Black- 42,1363128
Mountain-200 Silver- 38,1339394
Mountain-200 Silver- 46,1301029
Mountain-200 Black- 38,1294854


- Complex but Flexibly Ranking Using Window Functions

In [0]:
query = """
SELECT *
FROM (
    SELECT
        p.product_name,
        SUM(f.sales_amount) AS total_renenue,
        ROW_NUMBER() OVER (ORDER BY SUM(f.sales_amount) DESC) AS rank_products
    FROM gold.fact_sales AS f
    LEFT JOIN gold.dim_products AS p
        ON p.product_key = f.product_key
    GROUP BY p.product_name
) AS t
WHERE rank_products <= 5
"""
df = spark.sql(query)

df.display()

product_name,total_renenue,rank_products
Mountain-200 Black- 46,1373454,1
Mountain-200 Black- 42,1363128,2
Mountain-200 Silver- 38,1339394,3
Mountain-200 Silver- 46,1301029,4
Mountain-200 Black- 38,1294854,5


####What are the 5 worst-performing products in terms of sales?

In [0]:
query = """
SELECT
    p.product_name,
    SUM(f.sales_amount) AS total_renenue
FROM gold.fact_sales AS f
LEFT JOIN gold.dim_products AS p
    ON p.product_key = f.product_key
GROUP BY p.product_name
ORDER BY total_renenue
LIMIT 5
"""
df = spark.sql(query)

df.display()

product_name,total_renenue
Racing Socks- L,2430
Racing Socks- M,2682
Patch Kit/8 Patches,6382
Bike Wash - Dissolver,7272
Touring Tire Tube,7440


####Find the top 10 customers who have generated the highest revenue

In [0]:
query = """
SELECT
    c.customer_key,
    c.first_name,
    c.last_name,
    SUM(f.sales_amount) AS total_revenue
FROM gold.fact_sales AS f
LEFT JOIN gold.dim_customers AS c
    ON c.customer_key = f.customer_key
GROUP BY
    c.customer_key,
    c.first_name,
    c.last_name
ORDER BY total_revenue DESC
LIMIT 10
"""
df = spark.sql(query)

df.display()

customer_key,first_name,last_name,total_revenue
1302,Nichole,Nara,13294
1133,Kaitlyn,Henderson,13294
1309,Margaret,He,13268
1132,Randall,Dominguez,13265
1301,Adriana,Gonzalez,13242
1322,Rosa,Hu,13215
1125,Brandi,Gill,13195
1308,Brad,She,13172
1297,Francisco,Sara,13164
434,Maurice,Shan,12914


####The 3 customers with the fewest orders placed

In [0]:
query = """
SELECT
    c.customer_key,
    c.first_name,
    c.last_name,
    COUNT(DISTINCT order_number) AS total_orders
FROM gold.fact_sales AS f
LEFT JOIN gold.dim_customers AS c
    ON c.customer_key = f.customer_key
GROUP BY
    c.customer_key,
    c.first_name,
    c.last_name
ORDER BY total_orders
LIMIT 3
"""
df = spark.sql(query)

df.display()

customer_key,first_name,last_name,total_orders
10154,James,Thompson,1
13189,Charles,Turner,1
17930,Marie,Fernandez,1
